# Confinement Mode Threshold Correlations

Plot each registered L-H and L-I threshold-power relation against density. Use the button groups to select the threshold output and density input, then tune the remaining inputs with sliders.


In [1]:
import numpy as np

from bokeh.io import output_notebook, show
from bokeh.layouts import column, gridplot, row
from bokeh.models import (
    CheckboxButtonGroup,
    ColumnDataSource,
    CustomJS,
    Div,
    HoverTool,
    Legend,
    LegendItem,
    Slider,
)
from bokeh.palettes import Category20, Turbo256
from bokeh.plotting import figure

from fusdb.registry import RELATIONS

output_notebook()

relations = [
    # FusDB/cfspopcon L-H and L-I threshold relations.
    {
        "name": "L-H transition threshold power",
        "output": "P_LH",
        "density": "n_e_avg",
        "formula": "0.0488 * n20**0.717 * B0**0.803 * A_p**0.941",
    },
    {
        "name": "L-H transition threshold power (Martin-Ryter)",
        "output": "P_LH",
        "density": "n_e_avg",
        "formula": "0.0488 * (ne_eff19 / 10.0)**0.717 * B0**0.803 * A_p**0.941 * (2.0 / afuel) * (ne_eff19 / n19)**2.0 * scale",
    },
    {
        "name": "L-I transition threshold power HubbardNF17",
        "output": "P_LI_thresh",
        "density": "n_e_avg",
        "formula": "0.162 * (n19 / 10.0) * B0**0.262 * A_p * scale",
    },
    {
        "name": "L-I transition threshold power AUG",
        "output": "P_LI_thresh",
        "density": "n_e_avg",
        "formula": "0.14 * (n19 / 10.0) * (B0 / 2.4)**0.39 * A_p * scale",
    },
    {
        "name": "L-I transition threshold power HubbardNF12",
        "output": "P_LI_thresh",
        "density": "n_e_avg",
        "formula": "2.11 * I_p_MA**0.94 * (n19 / 10.0)**0.65 * scale",
    },
    # PROCESS L-H threshold relations.
    {
        "name": "L-H threshold ITER-1996 nominal",
        "output": "P_LH",
        "density": "n_la",
        "formula": "0.45 * n20**0.75 * B0 * R**2",
    },
    {
        "name": "L-H threshold ITER-1996 upper",
        "output": "P_LH",
        "density": "n_la",
        "formula": "0.3960502816 * n20 * B0 * R**2.5",
    },
    {
        "name": "L-H threshold ITER-1996 lower",
        "output": "P_LH",
        "density": "n_la",
        "formula": "0.5112987149 * n20**0.5 * B0 * R**1.5",
    },
    {
        "name": "L-H threshold Snipes-1997 ITER",
        "output": "P_LH",
        "density": "n_la",
        "formula": "0.65 * n20**0.93 * B0**0.86 * R**2.15",
    },
    {
        "name": "L-H threshold Snipes-1997 kappa",
        "output": "P_LH",
        "density": "n_la",
        "formula": "0.42 * n20**0.80 * B0**0.90 * R**1.99 * kappa**0.76",
    },
    {
        "name": "L-H threshold Martin-2008 nominal",
        "output": "P_LH",
        "density": "n_la",
        "formula": "0.0488 * n20**0.717 * B0**0.803 * A_p**0.941 * (2.0 / afuel)",
    },
    {
        "name": "L-H threshold Martin-2008 upper",
        "output": "P_LH",
        "density": "n_la",
        "formula": "0.05166240355 * n20**0.752 * B0**0.835 * A_p**0.96 * (2.0 / afuel)",
    },
    {
        "name": "L-H threshold Martin-2008 lower",
        "output": "P_LH",
        "density": "n_la",
        "formula": "0.04609619059 * n20**0.682 * B0**0.771 * A_p**0.922 * (2.0 / afuel)",
    },
    {
        "name": "L-H threshold Snipes-2000 nominal",
        "output": "P_LH",
        "density": "n_la",
        "formula": "1.42 * n20**0.58 * B0**0.82 * R * a**0.81 * (2.0 / afuel)",
    },
    {
        "name": "L-H threshold Snipes-2000 upper",
        "output": "P_LH",
        "density": "n_la",
        "formula": "1.547 * n20**0.615 * B0**0.851 * R**1.089 * a**0.876 * (2.0 / afuel)",
    },
    {
        "name": "L-H threshold Snipes-2000 lower",
        "output": "P_LH",
        "density": "n_la",
        "formula": "1.293 * n20**0.545 * B0**0.789 * R**0.911 * a**0.744 * (2.0 / afuel)",
    },
    {
        "name": "L-H threshold Snipes-2000 closed divertor nominal",
        "output": "P_LH",
        "density": "n_la",
        "formula": "0.8 * n20**0.5 * B0**0.53 * R**1.51 * (2.0 / afuel)",
    },
    {
        "name": "L-H threshold Snipes-2000 closed divertor upper",
        "output": "P_LH",
        "density": "n_la",
        "formula": "0.867 * n20**0.561 * B0**0.588 * R**1.587 * (2.0 / afuel)",
    },
    {
        "name": "L-H threshold Snipes-2000 closed divertor lower",
        "output": "P_LH",
        "density": "n_la",
        "formula": "0.733 * n20**0.439 * B0**0.472 * R**1.433 * (2.0 / afuel)",
    },
    {
        "name": "L-H threshold Martin-2008 aspect nominal",
        "output": "P_LH",
        "density": "n_la",
        "formula": "0.0488 * n20**0.717 * B0**0.803 * A_p**0.941 * (2.0 / afuel) * aspect_factor",
    },
    {
        "name": "L-H threshold Martin-2008 aspect upper",
        "output": "P_LH",
        "density": "n_la",
        "formula": "0.05166240355 * n20**0.752 * B0**0.835 * A_p**0.96 * (2.0 / afuel) * aspect_factor",
    },
    {
        "name": "L-H threshold Martin-2008 aspect lower",
        "output": "P_LH",
        "density": "n_la",
        "formula": "0.04609619059 * n20**0.682 * B0**0.771 * A_p**0.922 * (2.0 / afuel) * aspect_factor",
    },
    # PROCESS L-I threshold relations.
    {
        "name": "L-I threshold Hubbard-2012 nominal",
        "output": "P_LI_thresh",
        "density": "n_la",
        "formula": "2.11 * I_p_MA**0.94 * n20**0.65",
    },
    {
        "name": "L-I threshold Hubbard-2012 upper",
        "output": "P_LI_thresh",
        "density": "n_la",
        "formula": "2.11 * I_p_MA**1.18 * n20**0.83",
    },
    {
        "name": "L-I threshold Hubbard-2012 lower",
        "output": "P_LI_thresh",
        "density": "n_la",
        "formula": "2.11 * I_p_MA**0.70 * n20**0.47",
    },
    {
        "name": "L-I threshold Hubbard-2017",
        "output": "P_LI_thresh",
        "density": "n_la",
        "formula": "0.162 * n20 * A_p * B0**0.26",
    },
]

missing = [spec["name"] for spec in relations if spec["name"] not in RELATIONS]
if missing:
    raise RuntimeError(f"Missing relation definitions: {missing}")

colors = list(Category20[20])
if len(relations) > len(colors):
    colors.extend(Turbo256[i] for i in np.linspace(0, 255, len(relations) - len(colors), dtype=int))


def aspect_correction(A):
    if A <= 2.7:
        return 0.098 * A / (1.0 - (2.0 / (1.0 + A)) ** 0.5)
    return 1.0


def base_inputs(n_abs):
    R = 3.3
    a = 1.1
    A = R / a
    B0 = 5.3
    I_p_MA = 8.7
    ne_min19 = 0.7 * I_p_MA**0.34 * B0**0.62 * a**-0.95 * (R / a) ** 0.4
    return {
        "n19": n_abs / 1.0e19,
        "n20": n_abs / 1.0e20,
        "n_e_avg": n_abs,
        "n_la": n_abs,
        "B0": B0,
        "R": R,
        "a": a,
        "A": A,
        "A_p": 4 * np.pi**2 * R * a,
        "kappa": 1.8,
        "afuel": 2.5,
        "I_p_MA": I_p_MA,
        "scale": 1.0,
        "ne_min19": ne_min19,
        "ne_eff19": max(n_abs / 1.0e19, ne_min19),
        "aspect_factor": aspect_correction(A),
    }


x = np.geomspace(3.0e17, 8.0e19, 320)
sources = []
for color, spec in zip(colors, relations):
    y0 = np.array([eval(spec["formula"], {"np": np}, base_inputs(float(n))) for n in x])
    source = ColumnDataSource(
        data={
            "x": x,
            "y": y0,
            "visible_y": y0,
            "color": [color] * len(x),
            "label": [spec["name"]] * len(x),
            "output": [spec["output"]] * len(x),
            "density": [spec["density"]] * len(x),
        }
    )
    sources.append(source)

p = figure(
    width=980,
    height=620,
    x_axis_type="log",
    y_axis_type="log",
    title="Confinement mode threshold correlations",
    x_axis_label="density [m^-3]",
    y_axis_label="threshold power [MW]",
    tools="pan,wheel_zoom,box_zoom,reset,save",
    active_scroll="wheel_zoom",
    sizing_mode="stretch_width",
)
p.add_tools(
    HoverTool(
        tooltips=[
            ("relation", "@label"),
            ("output", "@output"),
            ("density", "@density"),
            ("n [m^-3]", "@x{0.000e+0}"),
            ("P [MW]", "@y{0.000}"),
        ]
    )
)

renderers = []
legend_items = []
for color, spec, source in zip(colors, relations, sources):
    renderer = p.line(
        "x",
        "visible_y",
        source=source,
        line_width=2,
        color=color,
        muted_alpha=0.08,
        name=spec["name"],
    )
    renderers.append(renderer)
    legend_items.append(LegendItem(label=f"{spec['name']} ({spec['density']} -> {spec['output']})", renderers=[renderer]))

legend = Legend(
    items=legend_items,
    click_policy="hide",
    label_text_font_size="8pt",
    spacing=2,
    orientation="vertical",
)
p.add_layout(legend, "right")

output_labels = sorted({spec["output"] for spec in relations})
density_labels = sorted({spec["density"] for spec in relations})
output_buttons = CheckboxButtonGroup(labels=output_labels, active=list(range(len(output_labels))), sizing_mode="stretch_width")
density_buttons = CheckboxButtonGroup(labels=density_labels, active=list(range(len(density_labels))), sizing_mode="stretch_width")
sliders = {
    "B0": Slider(title="B0 [T]", start=0.1, end=15.0, value=5.3, step=0.1),
    "R": Slider(title="R [m]", start=0.2, end=12.0, value=3.3, step=0.1),
    "a": Slider(title="a [m]", start=0.05, end=5.0, value=1.1, step=0.05),
    "A_p": Slider(title="plasma area A_p [m2]", start=1.0, end=300.0, value=4 * np.pi**2 * 3.3 * 1.1, step=1.0),
    "kappa": Slider(title="kappa", start=0.8, end=3.0, value=1.8, step=0.05),
    "afuel": Slider(title="fuel mass [amu]", start=1.0, end=3.0, value=2.5, step=0.05),
    "A": Slider(title="aspect ratio A", start=1.1, end=8.0, value=3.0, step=0.05),
    "I_p_MA": Slider(title="Ip [MA]", start=0.1, end=25.0, value=8.7, step=0.1),
    "scale": Slider(title="threshold scalar", start=0.1, end=5.0, value=1.0, step=0.05),
}

specs = [{"formula": spec["formula"], "output": spec["output"], "density": spec["density"]} for spec in relations]
callback = CustomJS(
    args={
        "sources": sources,
        "renderers": renderers,
        "specs": specs,
        "output_buttons": output_buttons,
        "density_buttons": density_buttons,
        **sliders,
    },
    code="""
function threshold(spec, nAbs) {
    const R_value = R.value;
    const a_value = a.value;
    const A_value = A.value;
    const B0_value = B0.value;
    const I_p_MA_value = I_p_MA.value;
    const n19 = nAbs / 1.0e19;
    const n20 = nAbs / 1.0e20;
    const neMin19 = 0.7 * I_p_MA_value**0.34 * B0_value**0.62 * a_value**(-0.95) * (R_value / a_value)**0.4;
    const neEff19 = Math.max(n19, neMin19);
    const aspectFactor = A_value <= 2.7 ? 0.098 * A_value / (1.0 - (2.0 / (1.0 + A_value))**0.5) : 1.0;
    const vars = {
        n19: n19,
        n20: n20,
        n_e_avg: nAbs,
        n_la: nAbs,
        B0: B0_value,
        R: R_value,
        a: a_value,
        A: A_value,
        A_p: A_p.value,
        kappa: kappa.value,
        afuel: afuel.value,
        I_p_MA: I_p_MA_value,
        scale: scale.value,
        ne_min19: neMin19,
        ne_eff19: neEff19,
        aspect_factor: aspectFactor,
    };
    const keys = Object.keys(vars);
    const values = keys.map((key) => vars[key]);
    return Function(...keys, `return ${spec.formula};`)(...values);
}
const activeOutputs = new Set(output_buttons.active.map((index) => output_buttons.labels[index]));
const activeDensities = new Set(density_buttons.active.map((index) => density_buttons.labels[index]));
for (let i = 0; i < sources.length; i++) {
    const source = sources[i];
    const spec = specs[i];
    const data = source.data;
    const x = data.x;
    const y = data.y;
    const visibleY = data.visible_y;
    for (let j = 0; j < x.length; j++) {
        y[j] = threshold(spec, x[j]);
        visibleY[j] = y[j];
    }
    const outputOk = activeOutputs.has(spec.output);
    const densityOk = activeDensities.has(spec.density);
    renderers[i].visible = outputOk && densityOk;
    source.change.emit();
}
""",
)

for widget in [output_buttons, density_buttons, *sliders.values()]:
    widget.js_on_change("active" if isinstance(widget, CheckboxButtonGroup) else "value", callback)

note = Div(
    text=(
        "<b>Density convention:</b> the x-axis is absolute density in m^-3. "
        "Each curve maps that density onto its registered density input, while all "
        "other scalar inputs are controlled below."
    )
)


def labeled_row(label, *children):
    return row(
        Div(text=f"<b>{label}</b>", width=115, styles={"padding-top": "7px"}),
        *children,
        sizing_mode="stretch_width",
    )


controls = column(
    labeled_row("Output", output_buttons),
    labeled_row("Density", density_buttons),
    gridplot(
        [
            [sliders["B0"], sliders["R"], sliders["a"]],
            [sliders["A_p"], sliders["kappa"], sliders["afuel"]],
            [sliders["A"], sliders["I_p_MA"], sliders["scale"]],
        ],
        sizing_mode="stretch_width",
    ),
    sizing_mode="stretch_width",
)

show(column(note, p, controls, sizing_mode="stretch_width"))


Loading BokehJS ...